# Methodology Chapter Draft

Converted from the original Python workflow script so the dissertation repository uses notebook-based workflow artefacts.


In [ ]:
from pathlib import Path
import csv
import json

from docx import Document
from docx.enum.table import WD_ALIGN_VERTICAL
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor


ROOT = Path("/Users/lu_nanxi/CASA/Dissertation_Data")
CUTOFF_DIR = ROOT / "Vivacity_full_day_cutoff_20260526"
GATE_DIR = CUTOFF_DIR / "integrity_gate"
MODEL_DIR = CUTOFF_DIR / "modelling_ready"
DIAG_DIR = MODEL_DIR / "causal_diagnostics"
OUT = ROOT / "dissertation_methodology_chapter_draft_vivacity.docx"


def read_csv(path):
    with path.open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def set_cell_shading(cell, fill):
    tc_pr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:fill"), fill)
    tc_pr.append(shd)


def set_cell_margins(cell, top=80, start=110, bottom=80, end=110):
    tc = cell._tc
    tc_pr = tc.get_or_add_tcPr()
    tc_mar = tc_pr.first_child_found_in("w:tcMar")
    if tc_mar is None:
        tc_mar = OxmlElement("w:tcMar")
        tc_pr.append(tc_mar)
    for m, v in [("top", top), ("start", start), ("bottom", bottom), ("end", end)]:
        node = tc_mar.find(qn(f"w:{m}"))
        if node is None:
            node = OxmlElement(f"w:{m}")
            tc_mar.append(node)
        node.set(qn("w:w"), str(v))
        node.set(qn("w:type"), "dxa")


def set_table_borders(table):
    tbl_pr = table._tbl.tblPr
    borders = tbl_pr.first_child_found_in("w:tblBorders")
    if borders is None:
        borders = OxmlElement("w:tblBorders")
        tbl_pr.append(borders)
    for edge in ("top", "left", "bottom", "right", "insideH", "insideV"):
        elem = borders.find(qn(f"w:{edge}"))
        if elem is None:
            elem = OxmlElement(f"w:{edge}")
            borders.append(elem)
        elem.set(qn("w:val"), "single")
        elem.set(qn("w:sz"), "4")
        elem.set(qn("w:space"), "0")
        elem.set(qn("w:color"), "DADCE0")


def add_run(paragraph, text, bold=False, italic=False, size=11, color="000000"):
    run = paragraph.add_run(text)
    run.bold = bold
    run.italic = italic
    run.font.name = "Arial"
    run._element.rPr.rFonts.set(qn("w:eastAsia"), "Arial")
    run.font.size = Pt(size)
    run.font.color.rgb = RGBColor.from_string(color)
    return run


def add_para(doc, text):
    p = doc.add_paragraph()
    p.paragraph_format.space_after = Pt(8)
    p.paragraph_format.line_spacing = 1.15
    add_run(p, text)
    return p


def add_heading(doc, text, level=1):
    p = doc.add_paragraph()
    p.paragraph_format.space_before = Pt(18 if level == 1 else 12)
    p.paragraph_format.space_after = Pt(6)
    add_run(p, text, size=19 if level == 1 else 14)
    return p


def add_bullets(doc, items):
    for item in items:
        p = doc.add_paragraph(style="List Bullet")
        p.paragraph_format.space_after = Pt(4)
        p.paragraph_format.line_spacing = 1.15
        add_run(p, item)


def add_callout(doc, title, text):
    table = doc.add_table(rows=1, cols=1)
    set_table_borders(table)
    cell = table.rows[0].cells[0]
    set_cell_shading(cell, "F8F9FA")
    set_cell_margins(cell, top=120, start=140, bottom=120, end=140)
    p = cell.paragraphs[0]
    p.paragraph_format.space_after = Pt(0)
    add_run(p, title + ": ", bold=True, size=10)
    add_run(p, text, size=10)


def add_scheme_table(doc):
    rows = [
        ("12b", "Cronton Road / Sandy Lane", "March 2023", "2023-03-01", "Descriptive only", "No quality-passing pre-intervention treated baseline is available under the current gate."),
        ("12d", "Aigburth Road / Otterspool Promenade", "March 2023", "2023-03-01", "Exploratory model", "Sufficient treated/control coverage, but causal diagnostics are weak."),
        ("12e", "Park Road / Path off Park Road", "March 2023", "2023-03-01", "Descriptive only", "No reliable treated pre-intervention baseline under current assumptions."),
        ("12f", "Leasowe / Wallasey corridor", "June 2023", "2023-06-01", "Exploratory model", "Strongest current comparison, but event-study warning remains."),
        ("13", "Astmoor / Warrington Road", "March 2024", "2024-03-01", "Exploratory model", "Usable pre-period, but cyclist event-study warning remains."),
    ]
    table = doc.add_table(rows=1, cols=6)
    table.autofit = False
    widths = [0.5, 1.45, 0.85, 0.85, 1.05, 1.7]
    for i, width in enumerate(widths):
        table.columns[i].width = Inches(width)
    set_table_borders(table)
    headers = ["Scheme", "Location", "Installation month", "Model date", "Use", "Reason"]
    for i, header in enumerate(headers):
        cell = table.rows[0].cells[i]
        set_cell_shading(cell, "F1F3F4")
        set_cell_margins(cell)
        add_run(cell.paragraphs[0], header, bold=True, size=8.5)
    for row in rows:
        cells = table.add_row().cells
        for i, value in enumerate(row):
            set_cell_margins(cells[i])
            cells[i].vertical_alignment = WD_ALIGN_VERTICAL.CENTER
            p = cells[i].paragraphs[0]
            p.paragraph_format.space_after = Pt(0)
            if i in [0, 2, 3, 4]:
                p.alignment = WD_ALIGN_PARAGRAPH.CENTER
            add_run(p, value, size=8.1)


def add_gate_table(doc, gate):
    rows = [
        ("Full-day cutoff", gate["cutoff"]),
        ("Quality day rule", gate["quality_ok_day"]),
        ("First reliable date", gate["first_reliable_date_rule"]),
        ("Pragmatic causal control", gate["pragmatic_causal_control_rule"]),
        ("Strict causal control", gate["strict_causal_control_rule"]),
        ("Duplicate daily keys", str(gate["duplicate_daily_keys"])),
    ]
    table = doc.add_table(rows=1, cols=2)
    table.autofit = False
    table.columns[0].width = Inches(1.75)
    table.columns[1].width = Inches(4.65)
    set_table_borders(table)
    for i, header in enumerate(["Criterion", "Definition / result"]):
        cell = table.rows[0].cells[i]
        set_cell_shading(cell, "F1F3F4")
        set_cell_margins(cell)
        add_run(cell.paragraphs[0], header, bold=True, size=8.8)
    for row in rows:
        cells = table.add_row().cells
        for i, value in enumerate(row):
            set_cell_margins(cells[i])
            cells[i].vertical_alignment = WD_ALIGN_VERTICAL.CENTER
            p = cells[i].paragraphs[0]
            p.paragraph_format.space_after = Pt(0)
            add_run(p, value, size=8.2)


def add_model_table(doc):
    rows = [
        ("Outcome", "Active travel, pedestrian, and cyclist counts per observed countline-day."),
        ("Main model", "log1p(outcome) as a function of time, post period, post-period slope, treated-post terms, seasonality, and countline fixed effects."),
        ("Estimator", "Weighted OLS on weekly countline-level data, weighted by observed days."),
        ("Uncertainty", "Cluster-robust standard errors by countline."),
        ("Diagnostics", "Pre-trend tests and event-study bins relative to the intervention month."),
        ("Interpretation", "Relative treated-control trajectories, not definitive causal impacts."),
    ]
    table = doc.add_table(rows=1, cols=2)
    table.autofit = False
    table.columns[0].width = Inches(1.45)
    table.columns[1].width = Inches(4.95)
    set_table_borders(table)
    for i, header in enumerate(["Component", "Specification"]):
        cell = table.rows[0].cells[i]
        set_cell_shading(cell, "F1F3F4")
        set_cell_margins(cell)
        add_run(cell.paragraphs[0], header, bold=True, size=8.8)
    for row in rows:
        cells = table.add_row().cells
        for i, value in enumerate(row):
            set_cell_margins(cells[i])
            cells[i].vertical_alignment = WD_ALIGN_VERTICAL.CENTER
            p = cells[i].paragraphs[0]
            p.paragraph_format.space_after = Pt(0)
            add_run(p, value, size=8.2)


def add_diagnostic_table(doc, diag_rows):
    decisions = {
        "12d": "Weak causal support; use exploratory/descriptive wording.",
        "12f": "Partly credible but still exploratory.",
        "13": "Partly credible but still exploratory.",
    }
    table = doc.add_table(rows=1, cols=4)
    table.autofit = False
    widths = [0.65, 1.2, 1.35, 3.2]
    for i, width in enumerate(widths):
        table.columns[i].width = Inches(width)
    set_table_borders(table)
    headers = ["Scheme", "Pre-trend warnings", "Event-study warnings", "Decision"]
    for i, header in enumerate(headers):
        cell = table.rows[0].cells[i]
        set_cell_shading(cell, "F1F3F4")
        set_cell_margins(cell)
        add_run(cell.paragraphs[0], header, bold=True, size=8.5)
    for row in diag_rows:
        values = [
            row["scheme_id"],
            row["pretrend_warnings"],
            f"{row['pre_event_warnings']} pre; {row['post_event_flags']} post",
            decisions.get(row["scheme_id"], row["causal_readiness"]),
        ]
        cells = table.add_row().cells
        for i, value in enumerate(values):
            set_cell_margins(cells[i])
            cells[i].vertical_alignment = WD_ALIGN_VERTICAL.CENTER
            p = cells[i].paragraphs[0]
            p.paragraph_format.space_after = Pt(0)
            if i in [0, 1, 2]:
                p.alignment = WD_ALIGN_PARAGRAPH.CENTER
            add_run(p, value, size=8.2)


def build_doc():
    with (GATE_DIR / "vivacity_integrity_gate_summary.json").open(encoding="utf-8") as f:
        gate = json.load(f)
    diag_rows = read_csv(DIAG_DIR / "vivacity_causal_diagnostic_summary.csv")

    doc = Document()
    section = doc.sections[0]
    section.top_margin = Inches(1)
    section.bottom_margin = Inches(1)
    section.left_margin = Inches(1)
    section.right_margin = Inches(1)

    styles = doc.styles
    styles["Normal"].font.name = "Arial"
    styles["Normal"]._element.rPr.rFonts.set(qn("w:eastAsia"), "Arial")
    styles["Normal"].font.size = Pt(11)

    title = doc.add_paragraph()
    title.paragraph_format.space_after = Pt(3)
    add_run(title, "Draft Methodology Chapter: Vivacity Active Travel Analysis", size=25)

    subtitle = doc.add_paragraph()
    subtitle.paragraph_format.space_after = Pt(12)
    add_run(
        subtitle,
        "Exploratory matched-control interrupted time-series design with data integrity and causal diagnostics",
        size=11,
        color="555555",
    )

    add_heading(doc, "3.1 Research Design", 1)
    add_para(
        doc,
        "This dissertation uses an exploratory matched-control interrupted time-series design to examine walking and cycling trajectories around selected Liverpool City Region active travel schemes. The design compares treated Vivacity countlines associated with active travel interventions against matched non-intervention countlines, while explicitly testing whether the available data are strong enough to support causal interpretation.",
    )
    add_para(
        doc,
        "The approach remains exploratory because some sensors may begin producing reliable records after the confirmed scheme date, and candidate control sites require manual verification for absence of other interventions. The method therefore prioritises transparent data cleaning, careful inclusion rules, and diagnostic testing over over-strong causal attribution.",
    )

    add_heading(doc, "3.2 Study Sites and Scheme Inclusion", 1)
    add_para(
        doc,
        "Five scheme groups were initially considered using Liverpool City Region scheme records and Vivacity sensor metadata. The final modelling strategy separates schemes that can support exploratory treated-control comparison from schemes that can only support descriptive interpretation.",
    )
    add_scheme_table(doc)

    add_heading(doc, "3.3 Data Sources", 1)
    add_para(
        doc,
        "The primary data source is Vivacity automated countline data, exported as classified count records and cleaned into daily and weekly countline-level datasets. The main observed modes are pedestrians, cyclists, and motorised traffic classes. Active travel is defined as the sum of pedestrian and cyclist counts.",
    )
    add_para(
        doc,
        "Contextual data were joined at LSOA 2021 level. These included Index of Multiple Deprivation 2019 deciles, income and employment deprivation deciles, Census 2021 population density, household car availability, and walking/cycling commute shares. These variables were used both to describe the socio-economic context of treated sites and to support matched-control selection.",
    )
    add_bullets(
        doc,
        [
            "Vivacity countline data: daily pedestrian, cyclist, active travel, and motorised counts.",
            "Vivacity metadata: countline names, sensor locations, route type, and hardware identifiers.",
            "IMD 2019: deprivation, income, and employment deciles linked to LSOA 2021 geography.",
            "Census 2021: population density, car availability, and commuting mode context.",
            "LCR cycle index: retained as supplementary background context rather than the main causal variable.",
        ],
    )

    add_heading(doc, "3.4 Vivacity Cleaning and Full-Day Cutoff", 1)
    add_para(
        doc,
        "Raw Vivacity records were aggregated to daily countline-level observations. Directional records were combined within countline and date, and mode classes were summed into pedestrian, cyclist, active travel, and motorised totals. Data were filtered to complete days up to 26 May 2026 to avoid incomplete recent observations.",
    )
    add_para(
        doc,
        "Each daily row was assessed for availability, missingness, and error flags. Days with missing availability, minimum availability below 80%, or recorded data errors were not treated as reliable model observations. Duplicate daily countline keys were also checked before modelling.",
    )

    add_heading(doc, "3.5 Integrity Gate", 1)
    add_para(
        doc,
        "Before modelling, a formal integrity gate was applied. This gate determined whether each countline had a reliable observation period and whether it could be used for causal, descriptive, or excluded purposes. The gate is important because early zero or missing values may reflect sensor absence rather than genuine absence of walking or cycling.",
    )
    add_gate_table(doc, gate)
    add_para(
        doc,
        f"The integrity gate assessed {gate['all_countlines']} countlines, including {gate['treated_countlines']} treated and {gate['control_countlines']} control countlines. Thirteen controls met the pragmatic causal threshold, while only two controls met the stricter seasonal threshold. This limited control base is a central reason why the final analysis remains exploratory.",
    )

    add_heading(doc, "3.6 Matched-Control and Equity Context Procedure", 1)
    add_para(
        doc,
        "Treated and candidate control countlines were spatially joined to LSOA 2021 boundaries. Candidate controls were selected from non-intervention countlines with similar LSOA-level socio-economic and built-context variables, including IMD decile, income deprivation, population density, car availability, and baseline walking/cycling commute shares.",
    )
    add_para(
        doc,
        "The matched-control process was used to identify plausible comparison sites, but it does not by itself guarantee causal validity. All controls still require manual verification to confirm they were not affected by other transport, public-realm, or active travel interventions during the study period.",
    )
    add_callout(
        doc,
        "Equity interpretation",
        "The final causal sample is weighted toward more deprived LSOAs and includes limited affluent comparison evidence. The dissertation can therefore analyse equity context, but should not claim a robust deprived-versus-affluent causal effect.",
    )

    add_heading(doc, "3.7 Modelling Strategy", 1)
    add_para(
        doc,
        "For schemes 12d, 12f, and 13, weekly countline-level models were estimated separately for active travel, pedestrians, and cyclists. Daily observations were aggregated to weekly counts per observed countline-day to reduce short-term noise and account for occasional missing days.",
    )
    add_model_table(doc)
    add_para(
        doc,
        "The main model estimates whether treated countlines changed differently from controls after the intervention month. The two key terms are the immediate treated-control post-intervention level change and the additional treated-control post-intervention weekly slope change. Effects are interpreted as relative trajectory differences rather than definitive scheme impacts.",
    )

    add_heading(doc, "3.8 Causal Diagnostics", 1)
    add_para(
        doc,
        "To test whether stronger causal interpretation was plausible, two diagnostic checks were added. First, pre-trend models used only pre-intervention observations to test whether treated and control countlines were already moving differently before the intervention month. Second, event-study diagnostics estimated treated-control differences in relative-week bins, using the four weeks before intervention as the reference period.",
    )
    add_diagnostic_table(doc, diag_rows)
    add_para(
        doc,
        "The diagnostics show that none of the modelled schemes fully satisfies the conditions required for strong causal language. Scheme 12d fails pre-trend checks for active travel and pedestrians. Scheme 12f is the strongest comparison but still has an active-travel pre-event warning. Scheme 13 has better pre-period coverage but a cyclist pre-event warning. These diagnostics are retained as robustness evidence and as a basis for cautious interpretation.",
    )

    add_heading(doc, "3.9 Limitations of the Method", 1)
    add_bullets(
        doc,
        [
            "Confirmed scheme dates are encoded in the current workflow.",
            "Several sensors do not provide reliable pre-intervention treated data, limiting before/after inference.",
            "The matched control pool is small and not fully manually verified.",
            "The final causal sample is not balanced across deprived and affluent LSOAs.",
            "Countline data measure flows at specific points and cannot identify individual users, trip purpose, or route substitution.",
            "Census and IMD variables describe residential area context rather than the socio-economic identity of people passing each sensor.",
        ],
    )

    add_heading(doc, "3.10 Reproducibility", 1)
    add_para(
        doc,
        "All cleaning, inclusion, modelling, and diagnostic steps were saved as reproducible scripts and outputs within the dissertation data directory. The key reproducible stages are the Vivacity full-day cutoff, integrity gate, modelling input builder, exploratory model script, equity context summary, and causal diagnostics script.",
    )
    add_bullets(
        doc,
        [
            "Full-day cutoff: Vivacity_full_day_cutoff_20260526.",
            "Integrity gate: Vivacity_full_day_cutoff_20260526/integrity_gate.",
            "Modelling-ready data: Vivacity_full_day_cutoff_20260526/modelling_ready/tables.",
            "Exploratory models: Vivacity_full_day_cutoff_20260526/modelling_ready/models.",
            "Causal diagnostics: Vivacity_full_day_cutoff_20260526/modelling_ready/causal_diagnostics.",
        ],
    )

    add_heading(doc, "3.11 Methodological Conclusion", 1)
    add_para(
        doc,
        "The final methodology is a transparent exploratory evaluation framework rather than a definitive causal impact design. Its strength lies in combining sensor data cleaning, contextual joining, matched-control selection, a formal integrity gate, and explicit causal diagnostics. This allows the dissertation to present empirical trajectory comparisons while also critically assessing the limits of causal inference in opportunistic active travel monitoring data.",
    )

    doc.save(OUT)


if __name__ == "__main__":
    build_doc()
    print(OUT)
